# Results Explorer — Adequacy 2050 (FR-DE)
## Dual & Solution Analysis

This notebook loads `solution_2050.nc`, `dual_2050.nc`, and `input_dataset_2050.nc`
to diagnose price formation, capacity adequacy, and identify drivers of load shedding.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

RESULTS = Path("../results/diagnostics")
EXPORT  = Path("../results/export")
DEMAND  = Path("../results/hourly_demand")
FIGURES = Path("../results/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

VOLL = 30_000.0  # EUR/MWh (load shedding cost)

## 1. Load Datasets

In [ ]:
sol = xr.open_dataset(RESULTS / "solution_2050.nc")
dual = xr.open_dataset(RESULTS / "dual_2050.nc")
inp  = xr.open_dataset(RESULTS / "input_dataset_2050.nc")

print("=== Solution variables ===")
for v in sorted(sol.data_vars):
    print(f"  {v}: dims={sol[v].dims}, shape={sol[v].shape}")

print("\n=== Dual variables (first 30) ===")
for i, v in enumerate(sorted(dual.data_vars)):
    print(f"  {v}: dims={dual[v].dims}, shape={dual[v].shape}")
    if i >= 29:
        print(f"  ... ({len(dual.data_vars)} total)")
        break

print("\n=== Input coords ===")
for c in sorted(inp.coords):
    vals = inp.coords[c].values
    print(f"  {c}: {len(vals)} values — {list(vals[:5])}{'...' if len(vals)>5 else ''}")

## 2. Electricity Prices from Duals

In [ ]:
# The electricity balance constraint dual = marginal price of electricity
# Search for the balance constraint in duals
balance_candidates = [v for v in dual.data_vars if "balance" in v.lower()]
print("Balance-related duals:", balance_candidates)

# Also check for any variable with 'electricity' in dims
elec_candidates = [v for v in dual.data_vars 
                   if any("elec" in str(d).lower() or "resource" in str(d).lower() 
                          for d in dual[v].dims)]
print("\nElectricity/resource duals:", elec_candidates[:10])

In [ ]:
# Try the most likely balance constraint name
price_var = None
for candidate in balance_candidates:
    da = dual[candidate]
    if "area" in da.dims or "conversion_tech" not in da.dims:
        price_var = candidate
        break

if price_var is None and balance_candidates:
    price_var = balance_candidates[0]

if price_var:
    print(f"Using dual variable: {price_var}")
    print(f"  dims: {dual[price_var].dims}")
    print(f"  shape: {dual[price_var].shape}")
    prices_da = dual[price_var]
else:
    print("No balance variable found — falling back to exported CSV")
    prices_da = None

In [ ]:
# Always load the exported CSV for comparison / fallback
prices_csv = pd.read_csv(EXPORT / "prices_ramp_flex23pct.csv")
prices_csv["hour"] = prices_csv["hour"].astype(int)

for area in ["FR", "DE"]:
    p = prices_csv[prices_csv["area"] == area]["value"]
    print(f"\n{area} prices (from CSV):")
    print(f"  Mean:   {p.mean():>10,.1f} EUR/MWh")
    print(f"  Median: {p.median():>10,.1f}")
    print(f"  P95:    {p.quantile(0.95):>10,.1f}")
    print(f"  P99:    {p.quantile(0.99):>10,.1f}")
    print(f"  Max:    {p.max():>10,.1f}")
    print(f"  Hours >= VOLL: {(p >= VOLL-1).sum()}")

### 2.1 Price Duration Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, area in zip(axes, ["FR", "DE"]):
    p = prices_csv[prices_csv["area"] == area]["value"].sort_values(ascending=False).values
    hours = np.arange(1, len(p)+1)
    ax.plot(hours, p, lw=1.5)
    ax.set_xlabel("Hours (sorted)")
    ax.set_ylabel("Price (EUR/MWh)")
    ax.set_title(f"{area} — Price Duration Curve")
    ax.set_yscale("log")
    ax.axhline(VOLL, color="red", ls="--", alpha=0.5, label=f"VoLL = {VOLL:,.0f}")
    ax.legend()

plt.tight_layout()
plt.savefig(FIGURES / "price_duration_curves_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.2 Price Heatmaps (hour-of-day × day-of-year)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, area in zip(axes, ["FR", "DE"]):
    p = prices_csv[prices_csv["area"] == area].copy()
    p["day"] = p["hour"] // 24
    p["hod"] = p["hour"] % 24
    pivot = p.pivot_table(index="hod", columns="day", values="value", aggfunc="mean")
    
    # Cap for visualisation
    vmax = min(pivot.values.max(), 500)
    im = ax.imshow(pivot.values, aspect="auto", cmap="hot_r", vmin=0, vmax=vmax,
                   origin="lower", interpolation="nearest")
    ax.set_xlabel("Day of year")
    ax.set_ylabel("Hour of day")
    ax.set_title(f"{area} — Electricity Price (capped at {vmax:.0f} EUR/MWh)")
    plt.colorbar(im, ax=ax, label="EUR/MWh")

plt.tight_layout()
plt.savefig(FIGURES / "price_heatmap_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Capacity & Investment Analysis

In [ ]:
caps = pd.read_csv(EXPORT / "conversion_capacity_ramp_flex23pct.csv")
print("Installed capacities (MW):")
print(caps.pivot_table(index="name", columns="area", values="value", aggfunc="sum")
      .fillna(0).round(0).to_string())

# Check which techs hit their investment cap
print("\n--- Investment saturation check ---")
# H2 CCGT caps from constants
h2_caps = {"FR": 10_000, "DE": 20_000}
for _, row in caps[caps["name"] == "Hydrogen_power_plant"].iterrows():
    area = row["area"]
    invested = row["value"]
    cap = h2_caps.get(area, 15_000)
    pct = invested / cap * 100
    print(f"  {area} H2 CCGT: {invested:,.0f} / {cap:,.0f} MW ({pct:.1f}% of cap)")

## 4. Solution Variables — Dispatch & Storage

In [ ]:
# List solution variables that look like dispatch/generation
gen_vars = [v for v in sol.data_vars 
            if any(k in v.lower() for k in ["conversion", "generation", "dispatch", "power"])]
storage_vars = [v for v in sol.data_vars
                if any(k in v.lower() for k in ["storage", "soc", "charge", "state"])]
flow_vars = [v for v in sol.data_vars
             if any(k in v.lower() for k in ["transport", "flow", "link"])]
shed_vars = [v for v in sol.data_vars
             if any(k in v.lower() for k in ["shed", "spill", "flexibility"])]

print("Generation/conversion vars:", gen_vars)
print("Storage vars:", storage_vars)
print("Flow/transport vars:", flow_vars)
print("Shedding/spillage/flex vars:", shed_vars)
print("\nAll solution vars:", sorted(sol.data_vars))

In [ ]:
# Try to extract hourly dispatch by technology
# Common POMMES variable names
dispatch_var = None
for candidate in ["conversion_power", "conversion_generation", "power_conversion"]:
    if candidate in sol.data_vars:
        dispatch_var = candidate
        break

if dispatch_var is None:
    # Try partial match
    for v in sol.data_vars:
        da = sol[v]
        if "hour" in da.dims and "conversion_tech" in da.dims:
            dispatch_var = v
            break

if dispatch_var:
    dispatch = sol[dispatch_var]
    print(f"Dispatch variable: {dispatch_var}")
    print(f"  dims: {dispatch.dims}, shape: {dispatch.shape}")
    
    # Show annual generation by tech and area
    annual_gen = dispatch.sum(dim="hour") / 1e6  # TWh
    print("\nAnnual generation (TWh):")
    for area in dispatch.coords.get("area", dispatch.coords.get("conversion_tech", [])).values[:2]:
        pass
    print(annual_gen)
else:
    print("Could not find dispatch variable — listing all vars with 'hour' dim:")
    for v in sol.data_vars:
        if "hour" in sol[v].dims:
            print(f"  {v}: {sol[v].dims}")

## 5. Load Shedding Deep Dive

In [ ]:
# Load shedding from solution
ls_var = None
for candidate in ["load_shedding_power", "load_shedding", "shedding"]:
    if candidate in sol.data_vars:
        ls_var = candidate
        break
if ls_var is None:
    for v in sol.data_vars:
        if "shed" in v.lower():
            ls_var = v
            break

if ls_var:
    ls = sol[ls_var]
    print(f"Load shedding variable: {ls_var}, dims={ls.dims}")
    
    # Extract per-area
    for area_val in (ls.coords.get("area", ls.coords.get("resource", [None])).values
                     if "area" in ls.dims else [None]):
        if area_val is not None:
            ls_area = ls.sel(area=area_val)
        else:
            ls_area = ls
        vals = ls_area.values.flatten()
        vals = vals[~np.isnan(vals)]
        nonzero = vals[vals > 0.1]
        print(f"\n  {area_val if area_val else 'All'}:")
        print(f"    Hours with shedding: {len(nonzero)}")
        print(f"    Total ENS: {vals.sum()/1e3:.1f} GWh")
        print(f"    Peak shedding: {vals.max():,.0f} MW")
        if len(nonzero) > 0:
            print(f"    Avg during shedding: {nonzero.mean():,.0f} MW")
else:
    print("No load shedding variable found — check solution vars above")

In [ ]:
if ls_var and "area" in sol[ls_var].dims:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    for ax, area_val in zip(axes, ["FR", "DE"]):
        try:
            ls_area = sol[ls_var].sel(area=area_val)
            # Squeeze extra dims
            vals = ls_area.values
            while vals.ndim > 1:
                vals = vals.sum(axis=tuple(range(1, vals.ndim)))
            hours = np.arange(len(vals))
            
            ax.fill_between(hours, 0, vals/1e3, alpha=0.7, color="crimson")
            ax.set_ylabel("Load shedding (GW)")
            ax.set_title(f"{area_val} — Hourly Load Shedding")
            ax.set_xlim(0, 8760)
            
            # Add month labels
            month_starts = [0, 744, 1416, 2160, 2880, 3624, 4344, 5088, 5832, 6552, 7296, 8016]
            month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
            ax.set_xticks(month_starts)
            ax.set_xticklabels(month_names)
        except Exception as e:
            ax.text(0.5, 0.5, f"Error: {e}", transform=ax.transAxes, ha="center")
    
    plt.tight_layout()
    plt.savefig(FIGURES / "load_shedding_timeline.png", dpi=150, bbox_inches="tight")
    plt.show()

### 5.1 Dunkelflaute Identification

In [ ]:
# Cross-reference shedding hours with VRE availability
# Load VRE capacity factors from input dataset
cf_var = None
for candidate in ["conversion_availability", "availability", "capacity_factor"]:
    if candidate in inp.data_vars:
        cf_var = candidate
        break

if cf_var:
    cf = inp[cf_var]
    print(f"Availability variable: {cf_var}, dims={cf.dims}")
    
    # Get Solar and Wind CFs for DE
    try:
        techs = cf.coords["conversion_tech"].values
        vre_techs = [t for t in techs if any(k in t.lower() for k in ["solar", "wind", "pv"])]
        print(f"VRE techs: {vre_techs}")
        
        for tech in vre_techs:
            cf_tech = cf.sel(conversion_tech=tech)
            if "area" in cf_tech.dims:
                cf_tech = cf_tech.sel(area="DE")
            vals = cf_tech.values.flatten()
            vals = vals[~np.isnan(vals)]
            print(f"  {tech}: mean CF={vals.mean():.3f}, min={vals.min():.3f}, "
                  f"max={vals.max():.3f}, hours at 0={np.sum(vals<0.01)}")
    except Exception as e:
        print(f"  Error extracting CFs: {e}")
else:
    print("No availability variable in input dataset")
    print("Available:", sorted(inp.data_vars))

### 5.2 Residual Load vs Load Shedding

In [ ]:
# Build residual load = demand - VRE generation
# Load demand from CSV
import csv
from collections import defaultdict

demand_by_area = {}
with open(DEMAND / "hourly_electricity_demand.csv") as f:
    for row in csv.DictReader(f):
        if row["year_op"] == "2050" and row["component"] == "total":
            area = row["area"]
            demand_by_area.setdefault(area, []).append(float(row["load_mw"]))

print("Demand loaded:")
for area, vals in demand_by_area.items():
    print(f"  {area}: {len(vals)} hours, peak={max(vals):,.0f} MW, "
          f"annual={sum(vals)/1e6:.1f} TWh")

## 6. Dual Variable Explorer

In [ ]:
# Show all dual variables and their statistics
print(f"Total dual variables: {len(dual.data_vars)}")
print()

# Group by type
groups = defaultdict(list)
for v in sorted(dual.data_vars):
    da = dual[v]
    vals = da.values.flatten()
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        continue
    
    # Classify
    if "balance" in v.lower():
        group = "BALANCE (prices)"
    elif "ramp" in v.lower():
        group = "RAMPING"
    elif "storage" in v.lower() or "soc" in v.lower():
        group = "STORAGE"
    elif "transport" in v.lower() or "flow" in v.lower():
        group = "TRANSPORT"
    elif "capacity" in v.lower() or "invest" in v.lower():
        group = "CAPACITY/INVEST"
    else:
        group = "OTHER"
    
    groups[group].append((v, da.dims, len(vals), vals.min(), vals.max(), vals.mean()))

for group, items in sorted(groups.items()):
    print(f"\n{'='*60}")
    print(f"  {group} ({len(items)} variables)")
    print(f"{'='*60}")
    for name, dims, n, vmin, vmax, vmean in items[:10]:
        print(f"  {name}")
        print(f"    dims={dims}, n={n}")
        print(f"    min={vmin:.4f}, max={vmax:.4f}, mean={vmean:.4f}")
    if len(items) > 10:
        print(f"  ... and {len(items)-10} more")

### 6.1 Investment Constraint Duals (Shadow Prices of Capacity)

In [ ]:
# Investment constraint duals tell us the value of additional capacity
invest_duals = {v: dual[v] for v in dual.data_vars 
                if "invest" in v.lower() or "capacity" in v.lower()}

print(f"Investment/capacity duals: {len(invest_duals)}")
for name, da in sorted(invest_duals.items()):
    vals = da.values.flatten()
    vals = vals[~np.isnan(vals)]
    nonzero = vals[np.abs(vals) > 0.01]
    if len(nonzero) > 0:
        print(f"\n  {name}:")
        print(f"    dims={da.dims}")
        print(f"    non-zero values: {len(nonzero)}/{len(vals)}")
        print(f"    range: [{nonzero.min():.2f}, {nonzero.max():.2f}]")
        print(f"    mean (non-zero): {nonzero.mean():.2f}")

### 6.2 Ramping Constraint Duals

In [ ]:
ramp_duals = {v: dual[v] for v in dual.data_vars if "ramp" in v.lower()}

print(f"Ramping duals: {len(ramp_duals)}")
for name, da in sorted(ramp_duals.items()):
    vals = da.values.flatten()
    vals = vals[~np.isnan(vals)]
    nonzero = vals[np.abs(vals) > 0.01]
    if len(nonzero) > 0:
        print(f"\n  {name}:")
        print(f"    dims={da.dims}")
        print(f"    binding hours: {len(nonzero)}/{len(vals)} ({100*len(nonzero)/max(len(vals),1):.1f}%)")
        print(f"    shadow price range: [{nonzero.min():.1f}, {nonzero.max():.1f}] EUR/MWh")
    else:
        print(f"\n  {name}: never binding")

### 6.3 Interconnection Constraint Duals

In [ ]:
transport_duals = {v: dual[v] for v in dual.data_vars 
                   if "transport" in v.lower() or "flow" in v.lower() or "link" in v.lower()}

print(f"Transport duals: {len(transport_duals)}")
for name, da in sorted(transport_duals.items()):
    vals = da.values.flatten()
    vals = vals[~np.isnan(vals)]
    nonzero = vals[np.abs(vals) > 0.01]
    print(f"\n  {name}: dims={da.dims}")
    if len(nonzero) > 0:
        print(f"    congested hours: {len(nonzero)}/{len(vals)} ({100*len(nonzero)/max(len(vals),1):.1f}%)")
        print(f"    shadow price range: [{nonzero.min():.1f}, {nonzero.max():.1f}] EUR/MWh")
    else:
        print(f"    never congested")

## 7. Adequacy Diagnostic Summary

In [ ]:
# Combine all findings
print("=" * 70)
print("ADEQUACY DIAGNOSTIC SUMMARY")
print("=" * 70)

# Load adequacy dashboard
dash = pd.read_csv(EXPORT / "adequacy_dashboard_ramp_flex23pct.csv")
for _, row in dash.iterrows():
    c = row["Country"]
    print(f"\n{c}:")
    print(f"  LOLE:          {row['LOLE (h)']:>8} hours")
    print(f"  ENS:           {row['ENS (GWh)']:>8.1f} GWh")
    print(f"  Peak shedding: {row['Peak LS (MW)']:>8,.0f} MW")
    print(f"  Mean price:    {row['Mean price (EUR/MWh)']:>8.1f} EUR/MWh")
    print(f"  Hours at VoLL: {row['Hours at VoLL']:>8}")

# Capacity recap
print("\n" + "-" * 70)
print("CAPACITY RECAP")
caps = pd.read_csv(EXPORT / "conversion_capacity_ramp_flex23pct.csv")
for area in ["FR", "DE"]:
    area_caps = caps[caps["area"] == area]
    total = area_caps["value"].sum()
    disp = area_caps[area_caps["name"].isin(
        ["Gas", "Hydrogen_power_plant", "Battery_1h", "Battery_4h",
         "Reservoir_Hydro_Plant", "RoR_Hydro"])]["value"].sum()
    print(f"\n{area}: total={total:,.0f} MW, dispatchable={disp:,.0f} MW")
    for _, row in area_caps.iterrows():
        print(f"  {row['name']:30s}: {row['value']:>10,.0f} MW")

print("\n" + "-" * 70)
print("KEY FINDING:")
print("DE has 343h load shedding because:")
print("  1. CLEVER declares 0 TWh gas for DE in 2050 (phaseout by 2045)")
print("  2. H2 CCGT capped at 20 GW — optimizer invested the full cap")
print("  3. Total dispatchable ~80 GW barely covers 81 GW peak demand")
print("  4. During extended Dunkelflaute, batteries drain in ~3h,")
print("     leaving only ~25 GW firm gen vs 81 GW demand")
print("  5. FR→DE interconnection limited to 4 GW")
print()
print("RECOMMENDATION: Raise H2 CCGT investment cap for DE,")
print("  or add long-duration storage (e.g. hydrogen storage)")
print("  to bridge multi-day Dunkelflaute periods.")

## 8. Storage State-of-Charge Profiles

In [ ]:
# Look for storage SoC in solution
soc_var = None
for candidate in ["storage_soc", "soc", "state_of_charge", "storage_energy"]:
    if candidate in sol.data_vars:
        soc_var = candidate
        break
if soc_var is None:
    for v in sol.data_vars:
        if "storage" in v.lower() and "hour" in sol[v].dims:
            soc_var = v
            break

if soc_var:
    soc = sol[soc_var]
    print(f"SoC variable: {soc_var}, dims={soc.dims}")
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    for ax, area_val in zip(axes, ["FR", "DE"]):
        try:
            soc_area = soc.sel(area=area_val)
            techs = soc_area.coords.get("storage_tech", soc_area.coords.get("conversion_tech", None))
            if techs is not None:
                for tech in techs.values:
                    vals = soc_area.sel(storage_tech=tech).values.flatten()
                    vals = vals[~np.isnan(vals)]
                    if vals.max() > 0:
                        ax.plot(vals/1e3, label=tech, lw=0.8)
            ax.set_ylabel("SoC (GWh)")
            ax.set_title(f"{area_val} — Storage State of Charge")
            ax.legend()
        except Exception as e:
            ax.text(0.5, 0.5, f"Error: {e}", transform=ax.transAxes, ha="center")
    
    plt.tight_layout()
    plt.savefig(FIGURES / "storage_soc_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No SoC variable found in solution")
    print("Solution variables:", sorted(sol.data_vars))

## 9. Cross-Border Flows

In [ ]:
flow_var = None
for candidate in ["transport_power", "flow", "transport_flow"]:
    if candidate in sol.data_vars:
        flow_var = candidate
        break
if flow_var is None:
    for v in sol.data_vars:
        if "transport" in v.lower() and "hour" in sol[v].dims:
            flow_var = v
            break

if flow_var:
    flow = sol[flow_var]
    print(f"Flow variable: {flow_var}, dims={flow.dims}")
    
    vals = flow.values.flatten()
    vals = vals[~np.isnan(vals)]
    print(f"  Range: [{vals.min():,.0f}, {vals.max():,.0f}] MW")
    print(f"  Mean: {vals.mean():,.0f} MW")
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(vals/1e3, lw=0.5, alpha=0.7)
    ax.set_xlabel("Hour")
    ax.set_ylabel("Flow (GW)")
    ax.set_title("FR ↔ DE Interconnection Flow")
    ax.axhline(0, color="k", lw=0.5)
    ax.axhline(4, color="red", ls="--", alpha=0.5, label="Cap 4 GW")
    ax.axhline(-4, color="red", ls="--", alpha=0.5)
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES / "cross_border_flows_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No flow variable found")

## 10. Sensitivity Analysis: H2 CCGT Cap Impact

In [ ]:
# The optimizer invested the full 20 GW H2 cap for DE.
# If the investment dual is large and positive, more capacity would reduce cost.

print("=== What-if: Higher H2 CCGT cap for DE ===")
print()

# From the adequacy dashboard
ens_de = 4423.47  # GWh
lole_de = 343     # hours
peak_ls = 55573   # MW

# Average shedding during LOLE hours
avg_shed = ens_de * 1000 / lole_de  # MW
print(f"Current: LOLE={lole_de}h, ENS={ens_de:.0f} GWh, avg shedding={avg_shed:,.0f} MW")
print()

# With 20 GW more H2 (total 40 GW):
# Dispatchable would be ~100 GW vs 81 GW peak → likely adequate for most hours
# But Dunkelflaute with 0 VRE: 40+55+4.75 = ~100 GW which covers 81 GW peak
print("Estimate with 40 GW H2 CCGT:")
new_disp = 40000 + 55000 + 4753
print(f"  Total dispatchable: {new_disp:,} MW vs peak {80903:,} MW")
print(f"  Margin: +{new_disp - 80903:,} MW → likely LOLE ≈ 0")
print()
print("Estimate with 30 GW H2 CCGT:")
new_disp_30 = 30000 + 55000 + 4753
print(f"  Total dispatchable: {new_disp_30:,} MW vs peak {80903:,} MW")
print(f"  Margin: +{new_disp_30 - 80903:,} MW")
print(f"  Still positive → batteries provide buffer for the ~9 GW gap")
print(f"  But extended Dunkelflaute may still cause some shedding")
print()

# Check the H2 investment dual to see marginal value
print("\nCheck dual variable for H2 investment cap to confirm.")